In [3]:
import re
import math
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

In [4]:
# 1. Dataset: minimal 4 dokumen pendek (1-3 kalimat)
#    Tema: sistem komputer, jaringan, AI, sistem temu kembali
documents = [
    "Sistem komputer terdiri dari perangkat keras dan perangkat lunak yang bekerja sama.",
    "Jaringan komputer menghubungkan banyak komputer agar dapat bertukar data dengan cepat.",
    "Kecerdasan buatan memungkinkan komputer belajar dan membuat keputusan seperti manusia.",
    "Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan dengan cepat.",
]


In [5]:
# 2. Preprocessing sederhana: case folding + tokenisasi + stopwords removal
def simple_preprocess(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text_no_stopwords = stopword_remover.remove(text)
    tokens = text_no_stopwords.split()
    return tokens


preprocessed_docs = [simple_preprocess(doc) for doc in documents]
preprocessed_texts = [" ".join(tokens) for tokens in preprocessed_docs]

print("=" * 70)
print("HASIL PREPROCESSING SEDERHANA")
print("=" * 70)
for i, (orig, proc) in enumerate(zip(documents, preprocessed_texts), start=1):
    print(f"\nDokumen {i}")
    print(f"  Asli       : {orig}")
    print(f"  Preprocess : {proc}")


HASIL PREPROCESSING SEDERHANA

Dokumen 1
  Asli       : Sistem komputer terdiri dari perangkat keras dan perangkat lunak yang bekerja sama.
  Preprocess : sistem komputer terdiri perangkat keras perangkat lunak bekerja sama

Dokumen 2
  Asli       : Jaringan komputer menghubungkan banyak komputer agar dapat bertukar data dengan cepat.
  Preprocess : jaringan komputer menghubungkan banyak komputer dapat bertukar data cepat

Dokumen 3
  Asli       : Kecerdasan buatan memungkinkan komputer belajar dan membuat keputusan seperti manusia.
  Preprocess : kecerdasan buatan memungkinkan komputer belajar membuat keputusan manusia

Dokumen 4
  Asli       : Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan dengan cepat.
  Preprocess : sistem temu informasi membantu pengguna menemukan dokumen relevan cepat


In [6]:
# 3. Bag-of-Words (raw count) menggunakan CountVectorizer
count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(preprocessed_texts)
vocab = count_vectorizer.get_feature_names_out()

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vocab,
    index=[f"Dok {i+1}" for i in range(len(documents))],
)

print("\n" + "=" * 70)
print("BAG-OF-WORDS (RAW COUNT)")
print("=" * 70)
print(bow_df)



BAG-OF-WORDS (RAW COUNT)
       banyak  bekerja  belajar  bertukar  buatan  cepat  dapat  data  \
Dok 1       0        1        0         0       0      0      0     0   
Dok 2       1        0        0         1       0      1      1     1   
Dok 3       0        0        1         0       1      0      0     0   
Dok 4       0        0        0         0       0      1      0     0   

       dokumen  informasi  ...  memungkinkan  menemukan  menghubungkan  \
Dok 1        0          0  ...             0          0              0   
Dok 2        0          0  ...             0          0              1   
Dok 3        0          0  ...             1          0              0   
Dok 4        1          1  ...             0          1              0   

       pengguna  perangkat  relevan  sama  sistem  temu  terdiri  
Dok 1         0          2        0     1       1     0        1  
Dok 2         0          0        0     0       0     0        0  
Dok 3         0          0        0 

In [7]:
# 4. Perhitungan MANUAL: TF, DF, IDF, TF-IDF
N = len(preprocessed_docs)  # jumlah dokumen

# --- 4a. Term Frequency (TF) ---
# TF(t, d) = jumlah kemunculan term t di dokumen d / jumlah total term di d
tf_data = {}
for i, tokens in enumerate(preprocessed_docs):
    doc_name = f"Dok {i+1}"
    total_terms = len(tokens)
    tf_data[doc_name] = {}
    for term in vocab:
        count = tokens.count(term)
        tf_data[doc_name][term] = count / total_terms if total_terms else 0

tf_df = pd.DataFrame(tf_data).T  # baris = dokumen, kolom = term
tf_df = tf_df[vocab]

print("\n" + "=" * 70)
print("TERM FREQUENCY (TF) - MANUAL")
print("=" * 70)
print(tf_df.round(4))

# --- 4b. Document Frequency (DF) dan Inverse Document Frequency (IDF) ---
# DF(t) = jumlah dokumen yang mengandung term t
# IDF(t) = log(N / DF(t)) + 1   -> smoothing agar konsisten dengan sklearn (default: smooth_idf=True)
# sklearn menggunakan rumus: IDF(t) = ln((1 + N) / (1 + DF(t))) + 1
df_data = {}
idf_data = {}
for term in vocab:
    df_count = sum(1 for tokens in preprocessed_docs if term in tokens)
    df_data[term] = df_count
    idf_data[term] = math.log((1 + N) / (1 + df_count)) + 1  # smoothing seperti sklearn

df_series = pd.Series(df_data, name="DF")
idf_series = pd.Series(idf_data, name="IDF")

df_idf_table = pd.concat([df_series, idf_series], axis=1)

print("\n" + "=" * 70)
print("DOCUMENT FREQUENCY (DF) DAN INVERSE DOCUMENT FREQUENCY (IDF) - MANUAL")
print("(menggunakan rumus smoothing: IDF = ln((1+N)/(1+DF)) + 1, sama seperti sklearn)")
print("=" * 70)
print(df_idf_table.round(4))

# --- 4c. TF-IDF manual = TF x IDF, lalu dinormalisasi (L2 norm) agar sama
#     dengan output default TfidfVectorizer (norm='l2')
tfidf_manual_raw = tf_df.copy()
for term in vocab:
    tfidf_manual_raw[term] = tf_df[term] * idf_series[term]

# normalisasi L2 per baris (per dokumen)
def l2_normalize(row):
    norm = math.sqrt((row ** 2).sum())
    return row / norm if norm != 0 else row

tfidf_manual = tfidf_manual_raw.apply(l2_normalize, axis=1)

print("\n" + "=" * 70)
print("TF-IDF MANUAL (setelah normalisasi L2)")
print("=" * 70)
print(tfidf_manual.round(4))



TERM FREQUENCY (TF) - MANUAL
       banyak  bekerja  belajar  bertukar  buatan   cepat   dapat    data  \
Dok 1  0.0000   0.1111    0.000    0.0000   0.000  0.0000  0.0000  0.0000   
Dok 2  0.1111   0.0000    0.000    0.1111   0.000  0.1111  0.1111  0.1111   
Dok 3  0.0000   0.0000    0.125    0.0000   0.125  0.0000  0.0000  0.0000   
Dok 4  0.0000   0.0000    0.000    0.0000   0.000  0.1111  0.0000  0.0000   

       dokumen  informasi  ...  memungkinkan  menemukan  menghubungkan  \
Dok 1   0.0000     0.0000  ...         0.000     0.0000         0.0000   
Dok 2   0.0000     0.0000  ...         0.000     0.0000         0.1111   
Dok 3   0.0000     0.0000  ...         0.125     0.0000         0.0000   
Dok 4   0.1111     0.1111  ...         0.000     0.1111         0.0000   

       pengguna  perangkat  relevan    sama  sistem    temu  terdiri  
Dok 1    0.0000     0.2222   0.0000  0.1111  0.1111  0.0000   0.1111  
Dok 2    0.0000     0.0000   0.0000  0.0000  0.0000  0.0000   0.0000  


In [8]:
# 5. Perhitungan TF-IDF menggunakan scikit-learn (TfidfVectorizer)
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(preprocessed_texts)
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

tfidf_sklearn_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vocab,
    index=[f"Dok {i+1}" for i in range(len(documents))],
)

print("\n" + "=" * 70)
print("TF-IDF DARI SCIKIT-LEARN (TfidfVectorizer)")
print("=" * 70)
print(tfidf_sklearn_df.round(4))



TF-IDF DARI SCIKIT-LEARN (TfidfVectorizer)
       banyak  bekerja  belajar  bertukar  buatan   cepat   dapat    data  \
Dok 1  0.0000   0.3158   0.0000    0.0000  0.0000  0.0000  0.0000  0.0000   
Dok 2  0.3481   0.0000   0.0000    0.3481  0.0000  0.2745  0.3481  0.3481   
Dok 3  0.0000   0.0000   0.3674    0.0000  0.3674  0.0000  0.0000  0.0000   
Dok 4  0.0000   0.0000   0.0000    0.0000  0.0000  0.2746  0.0000  0.0000   

       dokumen  informasi  ...  memungkinkan  menemukan  menghubungkan  \
Dok 1   0.0000     0.0000  ...        0.0000     0.0000         0.0000   
Dok 2   0.0000     0.0000  ...        0.0000     0.0000         0.3481   
Dok 3   0.0000     0.0000  ...        0.3674     0.0000         0.0000   
Dok 4   0.3483     0.3483  ...        0.0000     0.3483         0.0000   

       pengguna  perangkat  relevan    sama  sistem    temu  terdiri  
Dok 1    0.0000     0.6315   0.0000  0.3158  0.2490  0.0000   0.3158  
Dok 2    0.0000     0.0000   0.0000  0.0000  0.0000  0.00

In [9]:
# 6. Perbandingan manual vs scikit-learn
print("\n" + "=" * 70)
print("PERBANDINGAN: SELISIH ABSOLUT (MANUAL vs SCIKIT-LEARN)")
print("=" * 70)
diff_df = (tfidf_manual[tfidf_vocab] - tfidf_sklearn_df).abs()
print(diff_df.round(6))
print(f"\nSelisih maksimum antara hasil manual dan scikit-learn: "
      f"{diff_df.values.max():.6f}")
print("(Selisih mendekati 0 menunjukkan hasil perhitungan manual konsisten "
      "dengan scikit-learn.)")



PERBANDINGAN: SELISIH ABSOLUT (MANUAL vs SCIKIT-LEARN)
       banyak  bekerja  belajar  bertukar  buatan  cepat  dapat  data  \
Dok 1     0.0      0.0      0.0       0.0     0.0    0.0    0.0   0.0   
Dok 2     0.0      0.0      0.0       0.0     0.0    0.0    0.0   0.0   
Dok 3     0.0      0.0      0.0       0.0     0.0    0.0    0.0   0.0   
Dok 4     0.0      0.0      0.0       0.0     0.0    0.0    0.0   0.0   

       dokumen  informasi  ...  memungkinkan  menemukan  menghubungkan  \
Dok 1      0.0        0.0  ...           0.0        0.0            0.0   
Dok 2      0.0        0.0  ...           0.0        0.0            0.0   
Dok 3      0.0        0.0  ...           0.0        0.0            0.0   
Dok 4      0.0        0.0  ...           0.0        0.0            0.0   

       pengguna  perangkat  relevan  sama  sistem  temu  terdiri  
Dok 1       0.0        0.0      0.0   0.0     0.0   0.0      0.0  
Dok 2       0.0        0.0      0.0   0.0     0.0   0.0      0.0  
Dok 3 

In [10]:
# 7. Analisis: term dengan bobot TF-IDF tertinggi per dokumen
print("\n" + "=" * 70)
print("TERM DENGAN BOBOT TF-IDF TERTINGGI PER DOKUMEN (scikit-learn)")
print("=" * 70)

top_terms = {}
for doc_name in tfidf_sklearn_df.index:
    row = tfidf_sklearn_df.loc[doc_name]
    top_term = row.idxmax()
    top_score = row.max()
    top_terms[doc_name] = (top_term, top_score)
    print(f"{doc_name}: '{top_term}' (skor = {top_score:.4f})")


TERM DENGAN BOBOT TF-IDF TERTINGGI PER DOKUMEN (scikit-learn)
Dok 1: 'perangkat' (skor = 0.6315)
Dok 2: 'komputer' (skor = 0.4444)
Dok 3: 'belajar' (skor = 0.3674)
Dok 4: 'dokumen' (skor = 0.3483)


ANALISIS SINGKAT:
Term dengan bobot TF-IDF tertinggi pada tiap dokumen umumnya adalah term
yang sering muncul pada dokumen tersebut (TF tinggi) namun jarang muncul
pada dokumen lain (IDF tinggi), sehingga term tersebut bersifat unik dan
sangat merepresentasikan isi/topik dokumen. Sebaliknya, term yang muncul
di hampir semua dokumen (seperti "komputer" yang muncul di beberapa
dokumen bertema teknologi) akan memiliki IDF yang lebih rendah karena
kurang mampu membedakan satu dokumen dengan dokumen lainnya. Term-term
dengan skor TF-IDF tinggi inilah yang penting karena memberikan daya
pembeda (discriminative power) terbesar saat sistem IR melakukan
pencocokan antara query pengguna dengan dokumen, sehingga hasil pencarian
menjadi lebih relevan dan akurat. 